In [10]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np
import warnings

import cartopy.crs as ccrs
import cartopy.mpl.ticker as cticker
import cartopy.feature as cfeature


import proplot as pplt

from sklearn.cluster import KMeans

from eofs.multivariate.standard import MultivariateEof
from eofs.xarray import Eof

warnings.filterwarnings("ignore")

In [11]:
def daily_climo(da,varname,**kwargs):
  
    # This function is adapted the code written by Ray Bell for the SubX project; it is for the
    # verification data
    
    clim_fname = kwargs.get('fname', None)
    
    # Average daily data
    da_day_clim = da.groupby('time.dayofyear').mean('time')
    
    # Rechunk for time
    da_day_clim = da_day_clim.chunk({'dayofyear': 366})
    
    # Pad the daily climatolgy with nans
    x = np.empty((366, len(da_day_clim.lat), len(da_day_clim.lon)))
    x.fill(np.nan)
    _da = xr.DataArray(x,name=varname, coords=[np.linspace(1, 366, num=366, dtype=np.int64),
                              da_day_clim.lat, da_day_clim.lon],
                              dims = da_day_clim.dims)
    da_day_clim_wnan = da_day_clim.combine_first(_da)

    
    # Period rolling twice to make it triangular smoothing
    # See https://bit.ly/2H3o0Mf
    da_day_clim_smooth = da_day_clim_wnan.copy()
 
    

    for i in range(2):
        # Extand the DataArray to allow rolling to do periodic
        da_day_clim_smooth = xr.concat([da_day_clim_smooth[-15:],
                                        da_day_clim_smooth,
                                        da_day_clim_smooth[:15]],
                                        'dayofyear')
        # Rolling mean
        da_day_clim_smooth = da_day_clim_smooth.rolling(dayofyear=31,
                                                        center=True,
                                                        min_periods=1).mean()
        # Drop the periodic boundaries
        da_day_clim_smooth = da_day_clim_smooth.isel(dayofyear=slice(15, -15))

    
    # Extract the original days
    da_day_clim_smooth = da_day_clim_smooth.sel(dayofyear=da_day_clim.dayofyear)

    da_day_clim_smooth.name=varname
    ds_day_clim_smooth=da_day_clim_smooth.to_dataset()
    
    # Save to file if filename provide and return True, otherwise return the data
    if (clim_fname):
        ds_day_clim_smooth.to_netcdf(clim_fname)
        return True
    else:
        return ds_day_clim_smooth

In [12]:
from dask.distributed import Client
from dask.distributed import LocalCluster
cluster = LocalCluster()
cluster

LocalCluster(e2435aab, 'tcp://127.0.0.1:45345', workers=13, threads=104, memory=1.97 TiB)

### Define months, years and NA region as defined by Molina et al., 2023: https://journals.ametsoc.org/view/journals/aies/2/2/AIES-D-22-0051.1.xml

In [13]:
# Region
min_lat = 10
max_lat = 70
#min_lon = 150 #150 E
min_lon = 360-150 #150W
max_lon = 360-40 #40 W

# Date
#sdate = '1979-01-01'
#edate = '2019-12-31'
#sdate = '1999-01-01'
#edate = '2019-12-31'
sdate = '1981-01-01'
edate = '2019-12-31'


# Month
seas='DJF'
seas_mon=[1,2,3,12]
#seas='SONDJFM'
#seas_mon=[9,10,11,12,1,2,3]

npcs = 12

data_path = '/data/esplab/scratch/kpegion/obs_seus/wr_z500/'

In [14]:
inpath = '/data/esplab/shared/reanalysis/era5/daily/z500/'
ifname = 'z.*.nc'
ds_z = xr.open_mfdataset(inpath+ifname,combine='nested',concat_dim='time')
ds_z = ds_z.rename({'latitude':'lat','longitude':'lon'})
ds_z = ds_z.reindex(lat=list(reversed(ds_z['lat'])))
ds_z['z']=ds_z['z']/9.81

In [15]:
ds_z

<xarray.Dataset> Size: 8GB
Dimensions:  (time: 30316, lon: 360, lat: 181)
Coordinates:
  * time     (time) datetime64[ns] 243kB 1940-01-01 1940-01-02 ... 2022-12-31
  * lon      (lon) float32 1kB 0.0 1.0 2.0 3.0 4.0 ... 356.0 357.0 358.0 359.0
  * lat      (lat) float32 724B -90.0 -89.0 -88.0 -87.0 ... 87.0 88.0 89.0 90.0
    level    int32 4B 500
Data variables:
    z        (time, lat, lon) float32 8GB dask.array<chunksize=(366, 181, 360), meta=np.ndarray>
Attributes:
    comments:      Daily data created from: mean of 4xdaily
    source:        Downloaded from Copernicus Data Store: https://cds.climate...
    CreationDate:  2023-07-19
    CreatedBy:     kpegion
    Source:        makeDaily.ipynb

In [ ]:
# Make Anoms
ds_climo=daily_climo(ds_z['z'],'z')
ds_anoms=ds_z['z'].groupby('time.dayofyear')-ds_climo

# 5-day running means
ds_zanom=ds_anoms.rolling(time=5,center=True).mean().dropna('time')

# Subset months
ds_zanom = ds_zanom.sel(time=ds_zanom['time.month'].isin(seas_mon))

# Select the dates
ds_zanom=ds_zanom.sel(time=slice(sdate,edate))

In [ ]:
ds_zanom

In [ ]:
# Read in KMEANS clusters
ds_cluster=xr.open_dataset(data_path+'kmeans_4cluster_'+seas+'_1981-2019_NA.nc')

In [ ]:
ds_cluster['cluster'].values.min()

In [ ]:
ds_cluster['cluster'].values.min()

In [ ]:
ds_cluster

In [ ]:
# Re-assign the time dimension to contain the cluster assigned to each day
ds_zanom['time']=ds_cluster['cluster']
ds_zanom=ds_zanom.rename({'time':'cluster'})
ds_zanom

In [ ]:
cluster_comp=ds_zanom.groupby('cluster').mean().compute()
cluster_comp

In [ ]:
cluster_comp.to_netcdf(data_path+'era5_cluster_comp_z_na_'+seas+'1981-2019.nc')

In [ ]:
plt.contourf(cluster_comp['z'].sel(cluster=1))
plt.colorbar()

In [ ]:
cluster4_freq=(ds_zanom.groupby('cluster').count())/len(ds_cluster['time'])
freq=np.round((cluster4_freq['z'][:,0,0].values)*100)

In [ ]:
nrows=2
ncols=2
lon_0=260
f, axs = pplt.subplots(nrows=nrows,ncols=2,span=False,proj='npstere',proj_kw={'lon_0': lon_0},
                           figsize=(8.5,11.0),sharex=False,sharey=False)
axs.format(abc=True, abcloc='l', abcstyle='(a)')
levs=np.arange(-90,100,10)
cmap='NegPos'
titles=['0: Alaskan Ridge/Pacific Ridge','1: Greenland High','2: Pacific Trough','3: West Coast High']
k=0
for i in np.arange(nrows):
    for j in np.arange(ncols):

        # Make a filled contour plot
        cs1=axs[i,j].contourf(cluster_comp['lon'].values,cluster_comp['lat'].values,cluster_comp['z'][k,:,:].values,
                        cmap=cmap,extend='both',levels=levs,suptitle='ERA5 Z500 Anomalies DJF 1981-2019',
                        labels_kw={'fontsize': 'large'})
        
        axs[i,j].format(title=titles[k]+' ('+str(freq[k])+'%)',
                  reso='lo', coastcolor='black',
                  coast=True, innerborders=False, grid=False, boundinglat=10)
        k=k+1
        
f.colorbar(cs1,loc='b',length=0.6,label="m")
f.savefig('era5_cluster_comp_z_na_'+seas+'1981-2019.png')

### CHIRPS Precip Data

In [ ]:
fname_precip='/data/esplab/shared/obs/gridded/atm/precip/daily/chirps-v2.0/p05/*'
ds_precip=xr.open_mfdataset(fname_precip,combine='by_coords')
ds_precip = ds_precip.rename({'latitude':'lat','longitude':'lon'})
ds_precip=ds_precip.assign_coords(lon=((ds_precip['lon'] + 360) % 360))
ds_precip=ds_precip.sortby(ds_precip['lon'])
ds_precip

In [ ]:
ds_precip_NA=ds_precip.sel(lat=slice(min_lat,max_lat),lon=slice(min_lon,max_lon))

In [ ]:
ds_precip_NA

In [ ]:
# Make Anoms
ds_pclimo=ds_precip_NA.sel(time=slice(sdate,edate)).groupby('time.dayofyear').mean()
ds_panom=ds_precip_NA.groupby('time.dayofyear')-ds_pclimo

# Select the dates
ds_panom=ds_panom.sel(time=slice(sdate,edate))

ds_panom = ds_panom.sel(time=ds_panom['time.month'].isin(seas_mon))

In [ ]:
ds_panom['time']=ds_cluster['cluster']
ds_panom=ds_panom.rename({'time':'cluster'})
ds_panom

In [ ]:
cluster_pcomp=ds_panom.groupby('cluster').mean().compute()
cluster_pcomp

In [ ]:
cluster_pcomp.to_netcdf(data_path+'era5_cluster_comp_p_na_'+seas+'1981-2019.nc')

In [ ]:
nrows=2
ncols=2
lon_0=260
f, axs = pplt.subplots(nrows=nrows,ncols=2,span=False,proj='pcarree',proj_kw={'lon_0': lon_0},
                           figsize=(11.0,8.5),sharex=False,sharey=False)
axs.format(abc=True, abcloc='l', abcstyle='(a)')
levs=np.arange(-1,1.2,0.2)
cmap='DryWet'
k=0
for i in np.arange(nrows):
    for j in np.arange(ncols):

        # Make a filled contour plot
        cs1=axs[i,j].contourf(cluster_pcomp['lon'].values,cluster_pcomp['lat'].values,cluster_pcomp['precip'][k,:,:].values,
                        cmap=cmap,extend='both',levels=levs,
                        labels_kw={'fontsize': 'large'})
        
        axs[i,j].format(title=titles[k],
                  reso='lo', coastcolor='black',suptitle='CHIRPS Global Precip Anomalies DJF 1981-2019',
                  coast=True, innerborders=False, grid=False, lonlim=(225,300), latlim=(20,50))
        axs[i,j].add_feature(cfeature.STATES,edgecolor='gray') 

        k=k+1
        
f.colorbar(cs1,loc='b',length=0.6,label="mm/day") 
f.savefig('chirps05_cluster_comp_p_na_'+seas+'1981-2019.png')

### CPC Precip Data

In [ ]:
fname_precip='/data/esplab/shared/obs/gridded/atm/precip/daily/CPC-UNI-GLOBAL/precip.*.nc'
ds_precip_cpc=xr.open_mfdataset(fname_precip,combine='by_coords')
ds_precip_cpc=ds_precip_cpc.reindex(lat=list(reversed(ds_precip_cpc['lat'])))

In [ ]:
ds_precip_cpc

In [ ]:
ds_precip_NA=ds_precip_cpc.sel(lat=slice(min_lat,max_lat),lon=slice(min_lon,max_lon))

In [ ]:
# Make Anoms
ds_pclimo_cpc=ds_precip_NA.sel(time=slice(sdate,edate)).groupby('time.dayofyear').mean()
ds_panom_cpc=ds_precip_NA.groupby('time.dayofyear')-ds_pclimo_cpc
ds_panom_cpc

In [ ]:
ds_panom_cpc=ds_panom_cpc.sel(time=slice(sdate,edate))
ds_panom_cpc = ds_panom_cpc.sel(time=ds_panom_cpc['time.month'].isin(seas_mon))
ds_panom_cpc

In [ ]:
ds_cluster

In [ ]:
ds_panom_cpc['time']=ds_cluster['cluster']
ds_panom_cpc=ds_panom_cpc.rename({'time':'cluster'})
ds_panom_cpc

In [ ]:
cluster_pcomp_cpc=ds_panom_cpc.groupby('cluster').mean().compute()
cluster_pcomp_cpc

In [ ]:
nrows=2
ncols=2
lon_0=260
f, axs = pplt.subplots(nrows=nrows,ncols=2,span=False,proj='pcarree',proj_kw={'lon_0': lon_0},
                           figsize=(11.0,8.5),sharex=False,sharey=False)
axs.format(abc=True, abcloc='l', abcstyle='(a)')
levs=np.arange(-1,1.2,0.2)
cmap='DryWet'
k=0
for i in np.arange(nrows):
    for j in np.arange(ncols):

        # Make a filled contour plot
        cs1=axs[i,j].contourf(cluster_pcomp_cpc['lon'].values,cluster_pcomp_cpc['lat'].values,cluster_pcomp_cpc['precip'][k,:,:].values,
                        cmap=cmap,extend='both',levels=levs,
                        labels_kw={'fontsize': 'large'})
        
        axs[i,j].format(title=titles[k],
                  reso='lo', coastcolor='black',suptitle='CPC-UNI Global Precip Anomalies by Weather Regime (mm/day)',
                  coast=True, innerborders=False, grid=False, lonlim=(225,300), latlim=(20,50))
        axs[i,j].add_feature(cfeature.STATES,edgecolor='gray') 

        k=k+1
        
f.colorbar(cs1,loc='b',length=0.6,label="") 
f.savefig('cpcuniglobal_cluster_comp_p_na_'+seas+'1981-2019.png')

2025-02-17 22:49:55,319 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
2025-02-17 22:49:55,319 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
2025-02-17 22:49:55,320 - distributed.nanny - ERROR - Worker process died unexpectedly
2025-02-17 22:49:55,320 - distributed.nanny - ERROR - Worker process died unexpectedly
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
Traceback (most recent call last):
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
Process Dask Worker process (from Nanny):
  File "/home/aelyoussoufi/miniconda3/envs/latest/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/home/aelyoussoufi/miniconda3/envs/latest/lib/python3.10/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
2025-02-17 22:49:55,320 - distrib